# Watch parser, версия 3: последние четыре ошибки

V2 дала 83.3% exact и 46.7% error_acc. V2+guard — 93.3% exact и 86.7%
error_acc. Остались #24 («дважды в сутки»), #41/#42 (bad_ticker) и
#49 (дубль SBER).

V3 добавляет 126 точечных примеров: дубли тикеров, «дважды в сутки»,
`bad_ticker` с «Ромашкой»/«TSLA»/«Апельсином», больше `missing_*`/`ambiguous`.

Загрузите этот ноутбук в Colab, включите T4 и выполните ячейки по порядку.
Ноутбук попросит `watch_parser.zip`.

In [ ]:
!pip -q install -U "transformers>=4.56" "peft>=0.13" accelerate matplotlib
!pip -q uninstall -y torchao
# После этой ячейки: Среда выполнения → Перезапустить сеанс, затем выполнять со следующей.

In [ ]:
import glob
import os
import subprocess

from google.colab import files

os.chdir('/content')
project = '/content/finetuning/watch_parser'
v3_train = f'{project}/data/v3/train.jsonl'

if not os.path.exists(v3_train):
    print('Выберите файл watch_parser.zip')
    files.upload()
    zips = sorted(glob.glob('/content/*.zip'))
    if not zips:
        raise RuntimeError('zip не найден в /content')
    subprocess.run(['unzip', '-qo', zips[-1], '-d', '/content'], check=True)

os.chdir(project)
print('проект:', os.getcwd())
print('v3 train:', os.path.exists(v3_train))

In [ ]:
!python scripts/check_dataset.py --train data/v3/train.jsonl --valid data/v3/valid.jsonl --eval data/eval.jsonl --report data/v3/check_report.json
!python scripts/check_bot_compat.py --train data/v3/train.jsonl --valid data/v3/valid.jsonl --eval data/eval.jsonl

In [ ]:
!python training/train_lora.py \
  --train data/v3/train.jsonl \
  --valid data/v3/valid.jsonl \
  --out /content/adapter_watch_json_v3 \
  --run-dir /content/run_watch_json_v3

In [ ]:
# Модель без guard
!python scripts/eval_model.py \
  --adapter /content/adapter_watch_json_v3 \
  --prompt-mode fewshot \
  --out results/v3/tuned_prompt.json

# Модель + детерминированная страховка
!python scripts/eval_model.py \
  --adapter /content/adapter_watch_json_v3 \
  --prompt-mode fewshot --guard \
  --out results/v3/tuned_prompt_guard.json

In [ ]:
!python scripts/compare_runs.py \
  --base results/baseline/base_prompt.json \
  --run v1=results/v1/tuned_prompt.json \
  --run v2=results/v2/tuned_prompt.json \
  --run "v2+guard=results/v2/tuned_prompt_guard.json" \
  --run v3=results/v3/tuned_prompt.json \
  --run "v3+guard=results/v3/tuned_prompt_guard.json" \
  --out results/compare/compare_all.md

from IPython.display import Markdown, display

display(Markdown(open('results/compare/compare_all.md', encoding='utf-8').read()))

In [ ]:
!cd /content && zip -qr watch_results_v3.zip \
  finetuning/watch_parser/results \
  run_watch_json_v3 \
  adapter_watch_json_v3

from google.colab import files

files.download('/content/watch_results_v3.zip')